# Notebook 03 — Inter-Purchase Distribution Analysis

**Project:** Customer Segmentation and Retention Analysis Using Machine Learning and Statistical Modelling  
**Dataset:** UCI Online Retail II — Year 1 (December 2009 – December 2010)  
**Preceding notebooks:** `01_EDA_analysis.ipynb`, `02_Customer_feature_engineering.ipynb`

---

## Section A — Objective and Methodology

### A.1 What is an inter-purchase gap?

An **inter-purchase gap** (or inter-purchase interval) is the number of days between two consecutive valid purchase orders placed by the same customer. If a customer placed orders on 2 January, 28 January and 15 March, their gaps are 26 days and 46 days.

### A.2 Why does the gap distribution matter?

Fitting a probability distribution to the pool of observed gaps does several things:

1. **Summarises typical purchase timing** in a compact, interpretable form.
2. **Quantifies tail probabilities** — for any candidate inactivity threshold, we can calculate the model-implied probability that a healthy customer's next purchase would fall beyond that point purely by chance.
3. **Provides a principled inactivity threshold** instead of an arbitrary rule (e.g. 90 days). The threshold is grounded in what the data say about normal purchase rhythms.
4. **Connects to survival and retention modelling** — the shape parameter of the fitted distribution indicates whether the hazard of purchasing again is constant, increasing, or decreasing over time.

### A.3 Important conceptual distinctions

| Concept | Meaning |
| :--- | :--- |
| **Purchase gap** | Time between two observed consecutive purchases for one customer |
| **Inactivity** | A gap exceeding a chosen operational threshold; the customer is *at risk*, not confirmed lost |
| **Churn / confirmed lapse** | The customer never purchases again — this can only be established retrospectively and depends on observation-window assumptions |

> A gap exceeding the threshold tells us the customer is behaving unusually relative to their own historical pattern and the broader customer base. It does **not** confirm they will never return.

### A.4 Dataset scope and methodology

- **Data:** `online_retail_cleaned_data.csv` — all transactions with a valid `Customer ID` from Year 1 (Dec 2009 – Dec 2010).
- **Purchase events only:** Invoices beginning with `C` are cancellations and are excluded from purchase timing.
- **Invoice-level timestamps:** Each invoice is collapsed to its earliest line-item timestamp. Multiple product lines within the same invoice are *not* treated as separate purchase events.
- **Fitting datasets:** All valid inter-purchase intervals pooled across customers. Strictly positive intervals only (see Section B for zero-gap investigation).
- **Candidate distributions:** Exponential, Weibull (minimum), Gamma, Log-Normal — all fitted via maximum likelihood with location fixed at zero.
- **Model selection:** AIC and BIC, with visual diagnostic support (overlaid density curves, CDF comparison, Q-Q plots).
- **Threshold derivation:** Empirical 90th and 95th percentiles, cross-referenced with model-implied quantiles.

---
## Section B — Load and Validate the Data

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy import stats
from pathlib import Path

# ── Plotting style ──────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('Libraries loaded successfully.')

In [ ]:
# ── Project-relative path resolution (3-tier fallback) ─────────────────────
data_path = Path('../data/processed/online_retail_cleaned_data.csv')
if not data_path.exists():
    data_path = Path('data/processed/online_retail_cleaned_data.csv')
if not data_path.exists():
    data_path = Path('main_project/data/processed/online_retail_cleaned_data.csv')

df = pd.read_csv(data_path, parse_dates=['InvoiceDate'])

print(f'Loaded: {data_path}')
print(f'Shape:  {df.shape}')
print(f'Date range: {df["InvoiceDate"].min().date()} to {df["InvoiceDate"].max().date()}')
print(f'Unique customers: {df["Customer ID"].nunique():,}')
print(f'Missing Customer IDs: {df["Customer ID"].isna().sum()}')

In [ ]:
# ── 1. Separate valid purchases from cancellations ──────────────────────────
is_cancellation = df['Invoice'].astype(str).str.startswith('C')
purchase_df = df[~is_cancellation].copy()

print(f'Total rows:          {len(df):,}')
print(f'Purchase rows:       {len(purchase_df):,}')
print(f'Cancellation rows:   {is_cancellation.sum():,}')

# ── 2. Aggregate to one timestamp per (Customer, Invoice) ──────────────────
order_ts = (
    purchase_df
    .groupby(['Customer ID', 'Invoice'])['InvoiceDate']
    .min()
    .reset_index()
    .sort_values(['Customer ID', 'InvoiceDate'])
)

print(f'\nDistinct (customer, invoice) pairs: {len(order_ts):,}')
print(f'Purchase-eligible customers:        {order_ts["Customer ID"].nunique():,}')

In [ ]:
# ── 3. Compute successive inter-purchase intervals ─────────────────────────
order_ts['gap_days'] = (
    order_ts
    .groupby('Customer ID')['InvoiceDate']
    .diff()
    .dt.total_seconds() / 86400.0
)

# ── 4. Validation invariants ────────────────────────────────────────────────
all_gaps = order_ts['gap_days'].dropna()

assert (all_gaps < 0).sum() == 0, 'Negative gaps detected!'
assert np.isinf(all_gaps).sum() == 0, 'Infinite gaps detected!'

print(f'Total gap observations (incl. zero): {len(all_gaps):,}')
print(f'Customers contributing gaps:         {order_ts[order_ts["gap_days"].notna()]["Customer ID"].nunique():,}')
print(f'Negative gaps:                       {(all_gaps < 0).sum()}   OK')
print(f'Infinite gaps:                       {np.isinf(all_gaps).sum()}   OK')
print(f'Zero gaps:                           {(all_gaps == 0).sum()}')

In [ ]:
# ── 5. Investigate zero-day gaps ────────────────────────────────────────────
# Zero gaps arise when two distinct invoice numbers for the same customer
# share the identical earliest-line timestamp (placed within the same second).
# This represents a data-recording limitation, not two distinct purchase occasions.
zero_gap_rows = order_ts[order_ts['gap_days'] == 0.0]
print(f'Zero-gap occurrences: {len(zero_gap_rows)}')
print(f'Customers with same-second invoices: {zero_gap_rows["Customer ID"].nunique()}')
print('\nSample zero-gap invoices:')
print(zero_gap_rows[['Customer ID', 'Invoice', 'InvoiceDate', 'gap_days']].head(8).to_string(index=False))

In [ ]:
# ── 6. Final fitting dataset: strictly positive intervals only ─────────────
positive_gaps = all_gaps[all_gaps > 0].copy()
gaps_arr = positive_gaps.values
n = len(gaps_arr)

print(f'Gaps included in distribution fitting: {n:,}')
print(f'Zero-gap observations excluded:        {len(all_gaps) - n}')
print(f'Customers represented:                 {order_ts[order_ts["gap_days"] > 0]["Customer ID"].nunique():,}')

---
## Section C — Exploratory Analysis of Purchase Gaps

In [ ]:
# ── Descriptive statistics ──────────────────────────────────────────────────
desc = positive_gaps.describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
desc['skewness'] = positive_gaps.skew()
print('Inter-purchase gap (days) -- descriptive statistics\n')
print(desc.round(2).to_string())

**Interpretation:** The distribution is heavily right-skewed (skewness ~2.42). The median gap is approximately 21 days, while the mean is ~39 days -- a factor-of-two difference driven by a long upper tail. Most customers repurchase quickly; a minority generates very long intervals that inflate the average. The 90th percentile is ~102 days and the 95th percentile ~147 days, meaning about 5% of observed gaps exceed approximately five months.

In [ ]:
# ── Histogram with density scale ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(
    gaps_arr, bins=80, density=True,
    color='steelblue', alpha=0.7, edgecolor='none', label='Observed gaps',
)

ax.set_xlabel('Days between consecutive purchase orders', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Distribution of Inter-Purchase Gaps (Positive Intervals)', fontsize=13)
ax.set_xlim(0, 370)

for pct, label in [(0.50, 'Median\n21 d'), (0.90, 'P90\n102 d'), (0.95, 'P95\n147 d')]:
    val = np.quantile(gaps_arr, pct)
    ax.axvline(val, color='firebrick', lw=1.2, ls='--', alpha=0.8)
    ax.text(val + 2, ax.get_ylim()[1] * 0.85, label, color='firebrick', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Empirical CDF ────────────────────────────────────────────────────────────
sorted_gaps = np.sort(gaps_arr)
ecdf_y = np.arange(1, n + 1) / n

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(sorted_gaps, ecdf_y, lw=1.5, color='steelblue', label='ECDF')
ax.set_xlabel('Days between purchase orders', fontsize=12)
ax.set_ylabel('Cumulative probability', fontsize=12)
ax.set_title('Empirical CDF of Inter-Purchase Gaps', fontsize=13)
ax.set_xlim(0, 370)

for pct, label, col in [
    (0.90, 'P90 approx 102 d', 'firebrick'),
    (0.95, 'P95 approx 147 d', 'darkorange'),
]:
    val = np.quantile(gaps_arr, pct)
    ax.axhline(pct, color=col, ls='--', lw=1, alpha=0.8)
    ax.axvline(val, color=col, ls='--', lw=1, alpha=0.8)
    ax.text(val + 2, pct - 0.04, label, color=col, fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Boxplot ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.boxplot(
    gaps_arr, vert=False, widths=0.5, patch_artist=True,
    boxprops=dict(facecolor='lightsteelblue', color='steelblue'),
    medianprops=dict(color='firebrick', lw=2),
    flierprops=dict(marker='.', alpha=0.3, markersize=4, color='grey'),
)
ax.set_xlabel('Days between purchase orders', fontsize=12)
ax.set_title('Boxplot of Inter-Purchase Gaps', fontsize=13)
ax.set_yticks([])
plt.tight_layout()
plt.show()

**Interpretation:** The boxplot confirms the right-skewed, long-tailed nature of the data. The interquartile range runs roughly from 5 to 51 days. The many points plotted beyond the upper whisker (> ~130 d) show that very long gaps are not rare isolated events -- they form a meaningful tail representing customers with infrequent purchasing behaviour.

---
## Section D — Fit Candidate Distributions

We fit four continuous positive-support distributions using **maximum likelihood estimation** via `scipy.stats`. The location parameter is fixed at zero (`floc=0`) throughout, because purchase gaps must be strictly positive.

| Distribution | Parameters estimated (k) | Free parameters |
| :--- | :---: | :--- |
| **Exponential** | 1 | scale (= 1/lambda) |
| **Weibull (minimum)** | 2 | shape c, scale lambda |
| **Gamma** | 2 | shape a, scale theta |
| **Log-Normal** | 2 | log-scale sigma, scale (= exp(mu)) |

Model comparison uses:
- **AIC** = 2k - 2 * ln(L_hat)
- **BIC** = k * ln(n) - 2 * ln(L_hat)

In [ ]:
# ── Fit all four distributions ───────────────────────────────────────────────

# Exponential  (k=1)
exp_params  = stats.expon.fit(gaps_arr, floc=0)
exp_ll      = float(np.sum(stats.expon.logpdf(gaps_arr, *exp_params)))

# Weibull_min  (k=2)
weib_params = stats.weibull_min.fit(gaps_arr, floc=0)
weib_ll     = float(np.sum(stats.weibull_min.logpdf(gaps_arr, *weib_params)))

# Gamma        (k=2)
gam_params  = stats.gamma.fit(gaps_arr, floc=0)
gam_ll      = float(np.sum(stats.gamma.logpdf(gaps_arr, *gam_params)))

# Log-Normal   (k=2)
ln_params   = stats.lognorm.fit(gaps_arr, floc=0)
ln_ll       = float(np.sum(stats.lognorm.logpdf(gaps_arr, *ln_params)))

# Validate all fits are finite
for name, ll in [('Exponential', exp_ll), ('Weibull', weib_ll),
                  ('Gamma', gam_ll), ('Log-Normal', ln_ll)]:
    assert np.isfinite(ll), f'{name} log-likelihood is not finite!'
    print(f'{name:12s}: LL = {ll:,.2f}  OK')

In [ ]:
# ── Compute AIC and BIC ──────────────────────────────────────────────────────
fit_results = []
for name, k, ll, params in [
    ('Exponential',  1, exp_ll,  exp_params),
    ('Weibull',      2, weib_ll, weib_params),
    ('Gamma',        2, gam_ll,  gam_params),
    ('Log-Normal',   2, ln_ll,   ln_params),
]:
    aic = 2 * k - 2 * ll
    bic = k * np.log(n) - 2 * ll
    fit_results.append({
        'Distribution':    name,
        'k (free params)': k,
        'Log-Likelihood':  round(ll, 2),
        'AIC':             round(aic, 2),
        'BIC':             round(bic, 2),
        'params':          params,
    })

results_df = pd.DataFrame(fit_results).drop(columns='params')
results_df = results_df.sort_values('AIC').reset_index(drop=True)

print(f'n (gap observations) = {n:,}\n')
print(results_df.to_string(index=False))

In [ ]:
# ── Fitted parameter summary ─────────────────────────────────────────────────
print('Fitted Parameters (location fixed at 0 for all distributions):\n')

_, scale_e    = exp_params
c, _, scale_w = weib_params
a, _, scale_g = gam_params
s, _, scale_l = ln_params

param_rows = [
    {'Distribution': 'Exponential',  'Param 1': f'scale = {scale_e:.4f} d',      'Param 2': '--',                           'Implied mean': f'{scale_e:.2f} d'},
    {'Distribution': 'Weibull',      'Param 1': f'shape c = {c:.4f}',             'Param 2': f'scale = {scale_w:.4f} d',    'Implied mean': f'{stats.weibull_min.mean(*weib_params):.2f} d'},
    {'Distribution': 'Gamma',        'Param 1': f'shape a = {a:.4f}',             'Param 2': f'scale theta = {scale_g:.4f} d', 'Implied mean': f'{a*scale_g:.2f} d'},
    {'Distribution': 'Log-Normal',   'Param 1': f'sigma (log-scale) = {s:.4f}',  'Param 2': f'scale (exp mu) = {scale_l:.4f} d', 'Implied mean': f'{stats.lognorm.mean(*ln_params):.2f} d'},
]
print(pd.DataFrame(param_rows).to_string(index=False))

**Parameter interpretation:**

- **Exponential** (shape fixed at 1): Implies a memoryless process. The fitted mean of ~39 days matches the empirical mean.

- **Weibull** (shape c ~0.51 < 1): A shape below 1 indicates a *decreasing* hazard of repurchasing as time progresses -- once a customer has gone without buying for a while, they become progressively less likely to buy soon.

- **Gamma** (shape a ~0.38 < 1): Also indicates a decreasing hazard. The Gamma family allows more flexible tail behaviour than the Weibull. **Best AIC and BIC.**

- **Log-Normal** (sigma ~3.37): Very large log-scale standard deviation implies extreme tail heaviness. AIC/BIC are considerably worse here, and its model-implied P90 (~504 d) is absurd relative to the empirical value (~102 d).

---
## Section E — Compare the Fitted Distributions

In [ ]:
# ── Density overlay ───────────────────────────────────────────────────────────
x = np.linspace(0.01, 370, 1000)

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(gaps_arr, bins=80, density=True, color='lightsteelblue',
        alpha=0.55, edgecolor='none', label='Observed (density)')

ax.plot(x, stats.expon.pdf(x, *exp_params),        lw=1.8, ls=':',  color='grey',       label='Exponential')
ax.plot(x, stats.weibull_min.pdf(x, *weib_params),  lw=1.8, ls='--', color='darkorange',  label='Weibull')
ax.plot(x, stats.gamma.pdf(x, *gam_params),         lw=2.2, ls='-',  color='firebrick',   label='Gamma (best AIC/BIC)')
ax.plot(x, stats.lognorm.pdf(x, *ln_params),        lw=1.8, ls='-.', color='seagreen',    label='Log-Normal')

ax.set_xlim(0, 300)
ax.set_ylim(0, None)
ax.set_xlabel('Days between purchase orders', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Observed Gap Distribution vs. Fitted Density Curves', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── CDF comparison ───────────────────────────────────────────────────────────
sorted_gaps = np.sort(gaps_arr)
ecdf_y = np.arange(1, n + 1) / n

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(sorted_gaps, ecdf_y, lw=2, color='steelblue', label='ECDF')
ax.plot(x, stats.expon.cdf(x, *exp_params),        lw=1.6, ls=':',  color='grey',       label='Exponential')
ax.plot(x, stats.weibull_min.cdf(x, *weib_params),  lw=1.6, ls='--', color='darkorange',  label='Weibull')
ax.plot(x, stats.gamma.cdf(x, *gam_params),         lw=2.0, ls='-',  color='firebrick',   label='Gamma (best)')
ax.plot(x, stats.lognorm.cdf(x, *ln_params),        lw=1.6, ls='-.', color='seagreen',    label='Log-Normal')

ax.set_xlim(0, 300)
ax.set_xlabel('Days between purchase orders', fontsize=12)
ax.set_ylabel('Cumulative probability', fontsize=12)
ax.set_title('Empirical vs. Fitted CDFs', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Q-Q plots for Gamma and Weibull (the two best models) ────────────────────
theoretical_quantiles = np.linspace(1 / (n + 1), n / (n + 1), n)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, dist, params, col) in zip(axes, [
    ('Gamma',   stats.gamma,       gam_params,  'firebrick'),
    ('Weibull', stats.weibull_min,  weib_params, 'darkorange'),
]):
    theory_q = dist.ppf(theoretical_quantiles, *params)
    ax.scatter(theory_q, sorted_gaps, s=3, alpha=0.3, color=col)
    lo, hi = min(theory_q.min(), sorted_gaps.min()), max(theory_q.max(), sorted_gaps.max())
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='Perfect fit')
    ax.set_xlabel(f'Theoretical quantiles ({name})', fontsize=11)
    ax.set_ylabel('Observed quantiles (days)', fontsize=11)
    ax.set_title(f'Q-Q Plot: {name}', fontsize=12)
    ax.legend(fontsize=9)

plt.suptitle('Q-Q Plots for Best Candidate Distributions', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── KS statistics (for directional comparison only) ─────────────────────────
print('KS test D-statistic (p-values not interpretable with params fitted from same data):\n')
for name, dist, params in [
    ('Exponential', stats.expon,       exp_params),
    ('Weibull',     stats.weibull_min,  weib_params),
    ('Gamma',       stats.gamma,        gam_params),
    ('Log-Normal',  stats.lognorm,      ln_params),
]:
    ks_stat, _ = stats.kstest(gaps_arr, dist.cdf, args=params)
    print(f'  {name:12s}: KS D = {ks_stat:.4f}')

**Comparison findings:**

- **Gamma** achieves the **lowest AIC (123,760) and BIC (123,775)** -- a decisive margin over the next best (Weibull: AIC 126,370).
- **Weibull** is the second-best. Its shape c ~0.51 has the same qualitative implication as Gamma (decreasing hazard).
- **Exponential** performs noticeably worse -- it assumes a constant hazard rate and cannot capture the faster-than-exponential decay in the bulk.
- **Log-Normal** performs worst by a wide margin. It severely overestimates the probability of very long gaps, yielding model-implied P90 values of ~504 days -- physically impossible within a one-year window.
- **Q-Q plots:** Both Gamma and Weibull track the central bulk reasonably well but underestimate extreme upper quantiles (> ~250 d). This is a known limitation of parametric fits on long-tailed customer behaviour data.

> **Conclusion: The Gamma distribution is selected as the best candidate** for describing inter-purchase timing. It does not mean it is the true data-generating process -- it is the most defensible among the four tested.

---
## Section F — Derive and Justify an Inactivity Threshold

A **statistically justified inactivity threshold** answers: at what gap length does a customer's silence become unusual relative to the normal purchasing pattern of our customer base?

We evaluate both **empirical percentiles** (directly from the data, no distributional assumptions) and **model-implied quantiles** (from the fitted Gamma distribution).

### What this threshold is -- and is not

- It is an **operational definition of at-risk** -- not proof of permanent churn.
- Exceeding the threshold means the gap is longer than approximately 90% (or 95%) of all previously observed repeat-purchase intervals.
- This threshold is derived from historical intervals. It must **not** be used to define churn labels for predictive models trained on the same observation window -- doing so would introduce target leakage.
- Future retention prediction (Notebook 05+) will require an explicitly defined feature/target time-split.

In [ ]:
# ── Threshold candidate table ─────────────────────────────────────────────────
threshold_rows = []
for pct in [0.90, 0.95]:
    emp_val  = float(np.quantile(gaps_arr, pct))
    gam_val  = float(stats.gamma.ppf(pct, *gam_params))
    pct_exceed_emp = float((gaps_arr > emp_val).mean())
    gam_prob_exceed = float(1 - stats.gamma.cdf(emp_val, *gam_params))
    threshold_rows.append({
        'Percentile':                     f'P{int(pct*100)}',
        'Empirical threshold (d)':        round(emp_val, 1),
        'Gamma model threshold (d)':      round(gam_val, 1),
        'Obs exceeding empirical':        f'{pct_exceed_emp:.1%}',
        'Gamma P(gap > emp threshold)':   f'{gam_prob_exceed:.1%}',
    })

thresh_df = pd.DataFrame(threshold_rows)
print(thresh_df.to_string(index=False))

In [ ]:
# ── Visualise threshold on density curve ──────────────────────────────────────
x_plot = np.linspace(0.01, 370, 1000)

p90_emp = float(np.quantile(gaps_arr, 0.90))
p95_emp = float(np.quantile(gaps_arr, 0.95))

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(gaps_arr, bins=80, density=True, color='lightsteelblue',
        alpha=0.5, edgecolor='none', label='Observed')
ax.plot(x_plot, stats.gamma.pdf(x_plot, *gam_params), lw=2.5,
        color='firebrick', label='Gamma fit')

ax.axvspan(p90_emp, p95_emp, alpha=0.18, color='gold',   label='P90-P95 band')
ax.axvspan(p95_emp, 370,     alpha=0.12, color='salmon',  label='> P95 (extreme)')
ax.axvline(p90_emp, color='goldenrod', lw=1.8, ls='--', label=f'P90 = {p90_emp:.0f} d')
ax.axvline(p95_emp, color='tomato',    lw=1.8, ls='--', label=f'P95 = {p95_emp:.0f} d')

ax.set_xlim(0, 370)
ax.set_xlabel('Days between purchase orders', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Gamma Fit with Inactivity Threshold Candidates', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### Threshold comparison and recommendation

| Threshold | Empirical value | Gamma model value | Agreement |
| :--- | :--- | :--- | :--- |
| **P90** | ~102 days | ~111 days | Moderate (~9 d gap) |
| **P95** | ~147 days | ~165 days | Moderate (~18 d gap) |

The Gamma model consistently places its quantiles **slightly above** the empirical values. This is expected: the Gamma underestimates tail probabilities (visible in the Q-Q plot), so its implied quantiles for a given exceedance probability are slightly higher than the raw data.

**Recommendation: Use the empirical P90 ~102 days as the primary inactivity threshold.**

**Rationale:**
1. Directly grounded in the observed data, requiring no distributional assumption.
2. A 90th-percentile threshold means only the most unusual 10% of observed purchase intervals are flagged.
3. The Gamma model's P90 (~111 days) is reasonably close and provides mild supporting evidence.
4. A range of **90-111 days** can be presented to stakeholders as the evidence-based boundary zone.

**P95 (~147 days) as an alternative:** Appropriate if the goal is to flag only the most extreme laggards (top 5%) before triggering a retention campaign.

---
## Section G — Final Conclusions and Hand-Off

In [ ]:
import json

# ── Save distribution results CSV ────────────────────────────────────────────
save_results = []
for name, k, ll, params, dist_obj in [
    ('Exponential', 1, exp_ll,  exp_params,  stats.expon),
    ('Weibull',     2, weib_ll, weib_params, stats.weibull_min),
    ('Gamma',       2, gam_ll,  gam_params,  stats.gamma),
    ('Log-Normal',  2, ln_ll,   ln_params,   stats.lognorm),
]:
    aic = 2 * k - 2 * ll
    bic = k * np.log(n) - 2 * ll
    ks_stat, _ = stats.kstest(gaps_arr, dist_obj.cdf, args=params)
    p90_model = float(dist_obj.ppf(0.90, *params))
    p95_model = float(dist_obj.ppf(0.95, *params))
    save_results.append({
        'distribution':    name,
        'k_params':        k,
        'n_observations':  n,
        'log_likelihood':  round(ll, 4),
        'aic':             round(aic, 4),
        'bic':             round(bic, 4),
        'ks_statistic':    round(ks_stat, 6),
        'p90_model_days':  round(p90_model, 2) if np.isfinite(p90_model) else None,
        'p95_model_days':  round(p95_model, 2) if np.isfinite(p95_model) else None,
        'param1':          round(params[0], 6),
        'param2':          round(params[1], 6) if len(params) > 1 else None,
        'param3':          round(params[2], 6) if len(params) > 2 else None,
    })

out_path = Path('../data/processed/interpurchase_distribution_results.csv')
if not out_path.parent.exists():
    out_path = Path('data/processed/interpurchase_distribution_results.csv')
if not out_path.parent.exists():
    out_path = Path('main_project/data/processed/interpurchase_distribution_results.csv')

results_out_df = pd.DataFrame(save_results)
results_out_df.to_csv(out_path, index=False)
print(f'Distribution results saved: {out_path}')
print(results_out_df[['distribution','k_params','log_likelihood','aic','bic']].to_string(index=False))

In [ ]:
# ── Save threshold metadata JSON ──────────────────────────────────────────────
threshold_record = {
    'project':                 'Customer Segmentation and Retention Analysis',
    'notebook':                '03_Interpurchase_Distribution_Analysis',
    'dataset_scope':           'Year 2009-2010 (Dec 2009 to Dec 2010)',
    'n_gap_observations':      n,
    'n_customers_with_gaps':   int(order_ts[order_ts['gap_days'] > 0]['Customer ID'].nunique()),
    'best_distribution':       'Gamma',
    'gamma_shape':             round(gam_params[0], 6),
    'gamma_scale':             round(gam_params[2], 6),
    'selected_threshold_days': round(float(np.quantile(gaps_arr, 0.90)), 1),
    'threshold_basis':         'Empirical P90 of observed positive inter-purchase gaps',
    'threshold_range_days': {
        'empirical_p90': round(float(np.quantile(gaps_arr, 0.90)), 1),
        'gamma_p90':     round(float(stats.gamma.ppf(0.90, *gam_params)), 1),
        'empirical_p95': round(float(np.quantile(gaps_arr, 0.95)), 1),
        'gamma_p95':     round(float(stats.gamma.ppf(0.95, *gam_params)), 1),
    },
    'leakage_warning': (
        'Do NOT use this threshold to define churn labels for predictive models '
        'trained on the same observation window. Future retention prediction '
        'requires a separate feature/target time-split.'
    ),
}

json_path = Path('../data/processed/inactivity_threshold.json')
if not json_path.parent.exists():
    json_path = Path('data/processed/inactivity_threshold.json')
if not json_path.parent.exists():
    json_path = Path('main_project/data/processed/inactivity_threshold.json')

with open(json_path, 'w') as f:
    json.dump(threshold_record, f, indent=2)

print(f'Threshold metadata saved: {json_path}')
print(json.dumps(threshold_record, indent=2))

---
### Summary of Results

| Item | Value |
| :--- | :--- |
| **Valid gap observations fitted** | 14,831 (70 zero-second same-order pairs excluded) |
| **Customers contributing gaps** | 2,893 (repeat buyers; 1,421 single-order customers excluded) |
| **Best distribution (AIC and BIC)** | **Gamma** (AIC 123,760, BIC 123,775) |
| **Gamma parameters** | shape a = 0.378, scale theta = 103.4 d, loc = 0 (fixed) |
| **Second-best** | Weibull (AIC 126,370), with similar decreasing-hazard interpretation |
| **Empirical P50** | ~21 days |
| **Empirical P90** | ~**102 days** |
| **Empirical P95** | ~**147 days** |
| **Recommended threshold** | **102 days (empirical P90)** -- defensible range 102-111 days |

### Implications for K-Means customer segmentation (Notebook 04)

1. **Single-purchase customers (1,421):** Their `Median_InterPurchase_Days` feature is `NaN`. Must be handled in clustering (exclude gap feature for this cohort or use gap-agnostic features).
2. **Recency feature:** Customers with `Recency > 102` days are operating outside the normal re-purchase window and may form a distinct at-risk cluster.
3. **No standardisation yet:** The feature matrix must be scaled in Notebook 04 before K-Means fitting.

### Outstanding decisions before downstream modelling

| Decision | Status | Impact |
| :--- | :--- | :--- |
| Include Year 2 data? | Open | Would change all gap counts and possibly shift P90 threshold |
| Threshold value (P90 vs P95)? | Recommendation given (P90 ~102 d) | Affects how many customers are flagged as at-risk |
| How to handle single-order customers in K-Means? | Open | Rule-based cohort vs. RFM-only clustering |

---
*Notebook 03 complete. Ready to proceed to Notebook 04 -- Customer Segmentation (K-Means).*